# Deep Agents Package: Practice Exercise

Build a customer support system using the orchestrator pattern with specialized sub-agents. You will apply the same multi-agent architecture from the lesson to a new domain.

**What you'll implement:**
- System prompts for Product Agent, Order Agent, and Orchestrator
- Model selection for each agent
- Agent creation using `create_deep_agent`
- Sub-agent configuration for the orchestrator

**Provided for you:**
- Mock databases (products and orders)
- Product lookup tool (`get_product_info`)
- Order status tool (`get_order_status`)
- Helper function for streaming

**Estimated time:** 12-15 minutes

## Setup

Run the cell below to import all required packages and initialize the API clients.

In [ ]:
import os
import logging
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, Dict, Any

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_openai import ChatOpenAI

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

product_logger = logging.getLogger('product_agent')
order_logger = logging.getLogger('order_agent')
orchestrator_logger = logging.getLogger('orchestrator')

# Create shared workspace
workspace_dir = Path("./support_workspace")
workspace_dir.mkdir(exist_ok=True)

# Mock product database
PRODUCT_DATABASE = {
    "SKU-001": {
        "name": "Wireless Headphones Pro",
        "price": 149.99,
        "category": "Electronics",
        "in_stock": True,
        "description": "Premium noise-canceling wireless headphones with 30-hour battery life",
        "features": ["Active Noise Cancellation", "Bluetooth 5.0", "Foldable Design"]
    },
    "SKU-002": {
        "name": "Smart Watch Elite",
        "price": 299.99,
        "category": "Electronics",
        "in_stock": True,
        "description": "Advanced fitness tracking smartwatch with heart rate monitoring",
        "features": ["Heart Rate Monitor", "GPS Tracking", "Water Resistant"]
    },
    "SKU-003": {
        "name": "Ergonomic Keyboard",
        "price": 89.99,
        "category": "Accessories",
        "in_stock": False,
        "description": "Split ergonomic keyboard with mechanical switches",
        "features": ["Mechanical Switches", "Split Design", "Programmable Keys"]
    }
}

# Mock order database
ORDER_DATABASE = {
    "ORD-10001": {
        "customer_name": "Alice Johnson",
        "product_sku": "SKU-001",
        "status": "shipped",
        "tracking_number": "1Z999AA10123456784",
        "estimated_delivery": (datetime.now() + timedelta(days=2)).strftime("%Y-%m-%d"),
        "order_date": (datetime.now() - timedelta(days=3)).strftime("%Y-%m-%d")
    },
    "ORD-10002": {
        "customer_name": "Bob Smith",
        "product_sku": "SKU-002",
        "status": "processing",
        "tracking_number": None,
        "estimated_delivery": (datetime.now() + timedelta(days=5)).strftime("%Y-%m-%d"),
        "order_date": datetime.now().strftime("%Y-%m-%d")
    },
    "ORD-10003": {
        "customer_name": "Carol Davis",
        "product_sku": "SKU-003",
        "status": "delivered",
        "tracking_number": "1Z999AA10123456785",
        "estimated_delivery": (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d"),
        "order_date": (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
    }
}

print("Setup complete!")
print(f"Products in database: {list(PRODUCT_DATABASE.keys())}")
print(f"Orders in database: {list(ORDER_DATABASE.keys())}")

## Provided: Product Information Tool

This tool retrieves product details from the mock database. The Product Agent will use this tool to answer customer questions about products.

In [ ]:
def get_product_info(product_sku: str) -> str:
    """
    Retrieve detailed product information by SKU.
    
    Use this when customers ask about product details, pricing,
    availability, or features.
    
    Args:
        product_sku: The product SKU (e.g., "SKU-001", "SKU-002")
        
    Returns:
        str: Formatted product information including name, price,
             description, features, and stock status.
             Returns error message if SKU not found.
    """
    product_logger.info(f"Looking up product: {product_sku}")
    
    # Normalize the SKU
    sku = product_sku.strip().upper()
    
    # Look up the product
    if sku not in PRODUCT_DATABASE:
        return f"Product not found: {product_sku}. Please check the SKU and try again."
    
    product = PRODUCT_DATABASE[sku]
    
    # Format the response
    features_list = "\n".join(f"  - {f}" for f in product["features"])
    stock_status = "In Stock" if product["in_stock"] else "Out of Stock"
    
    return f"""Product: {product["name"]}
SKU: {sku}
Price: ${product["price"]:.2f}
Category: {product["category"]}

Description: {product["description"]}

Features:
{features_list}

Availability: {stock_status}"""

# Test the product tool
print("Test 1 - Valid SKU:")
print(get_product_info("SKU-001"))
print()
print("Test 2 - Invalid SKU:")
print(get_product_info("SKU-999"))

## Provided: Order Status Tool

This tool retrieves order status from the mock database. The Order Agent will use this tool to help customers track their orders.

In [ ]:
def get_order_status(order_id: str) -> str:
    """
    Retrieve order status and tracking information.
    
    Use this when customers ask about their order status,
    tracking information, or delivery estimates.
    
    Args:
        order_id: The order ID (e.g., "ORD-10001")
        
    Returns:
        str: Formatted order status including order date,
             current status, tracking number (if shipped),
             and estimated delivery date.
             Returns error message if order not found.
    """
    order_logger.info(f"Looking up order: {order_id}")
    
    # Normalize the order ID
    oid = order_id.strip().upper()
    
    # Look up the order
    if oid not in ORDER_DATABASE:
        return f"Order not found: {order_id}. Please check the order ID and try again."
    
    order = ORDER_DATABASE[oid]
    
    # Format tracking number
    tracking = order["tracking_number"] if order["tracking_number"] else "Not yet assigned"
    
    return f"""Order: {oid}
Customer: {order["customer_name"]}
Order Date: {order["order_date"]}

Status: {order["status"].upper()}
Tracking Number: {tracking}
Estimated Delivery: {order["estimated_delivery"]}"""

# Test the order tool
print("Test 1 - Shipped order:")
print(get_order_status("ORD-10001"))
print()
print("Test 2 - Processing order:")
print(get_order_status("ORD-10002"))
print()
print("Test 3 - Invalid order:")
print(get_order_status("ORD-99999"))

## Part 1: Create Specialized Sub-Agents

Create the Product Agent and Order Agent. For each agent, you need to:
1. Write a focused system prompt that defines the agent's expertise
2. Select an appropriate model
3. Create the agent using `create_deep_agent`

**Recall from the lesson:**
```python
research_system_prompt = """You are an expert research assistant...

YOUR EXPERTISE:
- Conduct thorough web research
- Synthesize information from multiple sources

TOOL USAGE:
- Use internet_search for all web research needs
...
"""

research_model = ChatOpenAI(model="gpt-4o", temperature=0.1)

research_agent = create_deep_agent(
    model=research_model,
    tools=[internet_search],
    system_prompt=research_system_prompt,
    backend=FilesystemBackend(root_dir=str(workspace_dir), virtual_mode=True)
)
```

**Tips for writing sub-agent prompts:**
- Define the agent's specific expertise clearly
- Explain when and how to use the available tool
- Specify what the agent should NOT handle (to maintain clear boundaries)

In [ ]:
# TODO: Write a system prompt for the Product Agent
# The Product Agent should:
# - Be an expert on product information (pricing, features, availability)
# - Use the get_product_info tool to look up products by SKU
# - NOT handle order-related queries

product_system_prompt = """
"""  # Write your prompt here


# TODO: Select a model for the Product Agent
product_model = None  # e.g., ChatOpenAI(model="gpt-4o", temperature=0.1)


# TODO: Create the Product Agent using create_deep_agent
# Use: model, tools=[get_product_info], system_prompt, backend with FilesystemBackend

product_agent = None  # Replace with your implementation


# Verify
if product_agent:
    print("Product Agent created!")
else:
    print("Please implement the Product Agent above")

In [ ]:
# TODO: Write a system prompt for the Order Agent
# The Order Agent should:
# - Be an expert on order tracking and status updates
# - Use the get_order_status tool to look up orders by order ID
# - NOT handle product information queries

order_system_prompt = """
"""  # Write your prompt here


# TODO: Select a model for the Order Agent
order_model = None  # e.g., ChatOpenAI(model="gpt-4o", temperature=0.1)


# TODO: Create the Order Agent using create_deep_agent
# Use: model, tools=[get_order_status], system_prompt, backend with FilesystemBackend

order_agent = None  # Replace with your implementation


# Verify
if order_agent:
    print("Order Agent created!")
else:
    print("Please implement the Order Agent above")

## Part 2: Configure the Orchestrator

Create the orchestrator that coordinates the sub-agents. You need to:
1. Write a system prompt that explains the orchestrator's role and available sub-agents
2. Select an appropriate model
3. Create the `subagent_list` configuration
4. Create the orchestrator using `create_deep_agent` with the `subagents` parameter

**Recall from the lesson:**
```python
orchestrator_system_prompt = """You are an orchestrator agent that coordinates specialized sub-agents.

YOUR ROLE:
- Analyze user requests and break them into sub-tasks
- Delegate sub-tasks to specialized agents

AVAILABLE SUB-AGENTS:
1. **Research Agent** (subagent_type='research')
   - Use for: Web research, information gathering
   
2. **Weather Agent** (subagent_type='weather')
   - Use for: Current weather, weather forecasts
...
"""

subagent_list = [
    {
        'name': 'research',
        'description': 'Expert research assistant for web search',
        'runnable': research_agent
    },
    {
        'name': 'weather', 
        'description': 'Weather specialist for conditions and forecasts',
        'runnable': weather_agent
    }
]

orchestrator = create_deep_agent(
    model=orchestrator_model,
    tools=[],  # No domain tools - only uses task tool for sub-agents
    system_prompt=orchestrator_system_prompt,
    backend=FilesystemBackend(root_dir=str(workspace_dir), virtual_mode=True),
    subagents=subagent_list
)
```

**Tips for writing orchestrator prompts:**
- Clearly define the orchestrator's coordination role
- List available sub-agents with their `subagent_type` names
- Describe when to use each sub-agent

In [ ]:
# TODO: Write a system prompt for the Orchestrator
# The Orchestrator should:
# - Coordinate the Product Agent and Order Agent
# - Analyze customer inquiries and delegate to the right specialist
# - Combine information from multiple agents when needed
# - List available sub-agents: 'product' for product queries, 'order' for order queries

orchestrator_system_prompt = """
"""  # Write your prompt here


# TODO: Select a model for the Orchestrator
orchestrator_model = None  # e.g., ChatOpenAI(model="gpt-4o", temperature=0.1)

In [ ]:
# TODO: Create the subagent_list configuration
# This should be a list of dictionaries, each with:
# - 'name': identifier for the sub-agent ('product' or 'order')
# - 'description': what the agent does
# - 'runnable': the agent object (product_agent or order_agent)

subagent_list = []  # Replace with your implementation


# TODO: Create the orchestrator using create_deep_agent
# Use:
# - model: orchestrator_model
# - tools: empty list [] (orchestrator uses task tool for sub-agents)
# - system_prompt: orchestrator_system_prompt
# - backend: FilesystemBackend with workspace_dir, virtual_mode=True
# - subagents: subagent_list

orchestrator = None  # Replace with your implementation


# Verify your orchestrator was created
if orchestrator and len(subagent_list) == 2:
    print("Orchestrator configured successfully!")
    print(f"Sub-agents registered: {[s['name'] for s in subagent_list]}")
else:
    print("Please implement the subagent_list and orchestrator above")

## Test Your Implementation

Run the test queries below to verify your customer support system works correctly.

In [ ]:
# Helper function (provided)
def stream_support(query: str):
    """Stream orchestrator execution for customer support queries."""
    print("=" * 80)
    print(f"CUSTOMER: {query}")
    print("=" * 80)
    print()
    
    seen_message_ids = set()
    
    for chunk in orchestrator.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="values"
    ):
        if "messages" in chunk:
            for message in chunk["messages"]:
                msg_id = message.id
                if msg_id in seen_message_ids:
                    continue
                seen_message_ids.add(msg_id)
                
                msg_type = getattr(message, 'type', 'unknown')
                
                if msg_type == 'ai' and hasattr(message, 'tool_calls') and message.tool_calls:
                    for tc in message.tool_calls:
                        tool_name = tc.get('name', 'unknown')
                        if tool_name == 'task':
                            subagent_type = tc.get('args', {}).get('subagent_type', 'unknown')
                            print(f"[ORCHESTRATOR] Delegating to {subagent_type} agent...")
                elif msg_type == 'ai' and message.content:
                    print(f"[SUPPORT] {message.content}")
                    print("-" * 80)
    
    print()
    print("=" * 80)

print("Helper function ready!")

In [ ]:
# Test 1: Product inquiry
stream_support("Can you tell me about the Wireless Headphones Pro (SKU-001)?")

In [ ]:
# Test 2: Order status inquiry
stream_support("What's the status of my order ORD-10001?")

In [ ]:
# Test 3: Combined inquiry (requires both agents)
stream_support(
    "I placed order ORD-10002. Can you check the status and also tell me "
    "more about the Smart Watch Elite (SKU-002) that I ordered?"
)